# WOA13 Temperature and Salinity Bias

Zonal average T/S bias relative to WOA13 observations in latitude-depth space.

Authors: John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variables: T and S on annual z-space grid
variables = [
    RequestedVariable("thetao", "ocean_annual_z", frequency="yearly"),
    RequestedVariable("so", "ocean_annual_z", frequency="yearly"),
]

# Diagnostic configuration
diag_name = "WOA13 T/S Bias"
diag_desc = "Zonal average T/S bias relative to WOA13 observations in y-z space"
user_options = {}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-1",     date_range=("0101-01-01", "0200-12-31"), name="CM4.0"),
    CaseGroup2("cmip6-1183", date_range=("0101-01-01", "0200-12-31"), name="ESM4.1"),
    CaseGroup2("esm45-122",  date_range=("0101-01-01", "0200-12-31"), name="ESM4.5 piControl"),
    CaseGroup2("esm45-148",  date_range=("0001-01-01", "0100-12-31"), name="ESM4.5 piControl branched"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

In [ ]:
convert_to_momgrid(diag, return_corners=True)

## Main Diagnostic

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import momgrid
import cmip_basins
import momgrid.VerticalSplitScale as VerticalSplitScale

os.environ["MOMGRID_WEIGHTS_DIR"] = "/nbhome/jpk/grid_weights"

In [ ]:
all_figs = []
climplot.publication()

In [ ]:
def yz_stats(arr, y, z_i):
    """Compute latitude/depth-weighted bias, min, max, RMSE.

    Parameters
    ----------
    arr : xarray.DataArray
        2-D field (z_l, yh) of bias values
    y : xarray.DataArray
        1-D latitude coordinate
    z_i : array-like
        1-D depth interfaces (length N+1)

    Returns
    -------
    stats : dict
        Dictionary with bias, min, max, rmse (4 sig figs)
    stats_str : str
        Formatted string for plot annotation
    """
    ybnds = momgrid.util.infer_bounds(y.values)
    dy = np.abs(ybnds[1:] - ybnds[:-1])
    dy = dy * (40075000 / 360)  # degrees to meters

    z_i_vals = z_i.values if hasattr(z_i, "values") else np.asarray(z_i)
    dz = np.abs(z_i_vals[1:] - z_i_vals[:-1])

    weight = np.multiply(*np.meshgrid(dy, dz))

    masked_arr = np.ma.array(arr.values, mask=np.isnan(arr.values))
    mean = np.ma.average(masked_arr, weights=weight)
    vmax = masked_arr.max()
    vmin = masked_arr.min()

    se = masked_arr * masked_arr
    mse = np.ma.average(se, weights=weight)
    rmse = np.sqrt(mse)

    stats = {
        "bias": f"{mean:.4g}",
        "min": f"{vmin:.4g}",
        "max": f"{vmax:.4g}",
        "rmse": f"{rmse:.4g}",
    }

    stats_str = (
        f"min= {vmin:.4g}  max= {vmax:.4g}\n"
        f"mean= {mean:.4g}  rmse={rmse:.4g}"
    )

    return (stats, stats_str)

In [ ]:
WOA13_OBS_PATHS = {
    "om4": "/archive/jpk/datasets/OM5/obs/WOA13/WOA13_z_35level_OM4_1080x1440_annual_v20250214.nc",
    "om5": "/archive/jpk/datasets/OM5/obs/WOA13/WOA13_z_35level_OM5_1161x1440_annual_v20240602.nc",
}

BASINS = {
    "Global": {"codes": None, "xlim": None},
    "Atlantic+Arctic": {"codes": [2, 4], "xlim": (-30, None)},
    "Indo-Pacific": {"codes": [3, 5], "xlim": (-35, None)},
}

In [ ]:
# Cache for WOA13 observations to avoid reloading
_woa13_cache = {}

def load_woa13(obs_key):
    """Load and cache WOA13 observations."""
    global _woa13_cache
    if obs_key not in _woa13_cache:
        _woa13_cache[obs_key] = xr.open_dataset(WOA13_OBS_PATHS[obs_key])
        _woa13_cache[obs_key] = momgrid.util.reset_nominal_coords(_woa13_cache[obs_key])
    return _woa13_cache[obs_key]


# Process each experiment: load obs, compute bias, generate basin masks
data = {}

for group in diag.groups:
    ds_t = group.datasets[diag.varmap["thetao"]]
    ds_s = group.datasets[diag.varmap["so"]]

    # Detect model type from attrs set by convert_to_momgrid
    model_type = ds_t.attrs.get("model", "")
    if "om4" in model_type:
        obs_key = "om4"
    elif "om5" in model_type:
        obs_key = "om5"
    else:
        raise ValueError(f"Model type '{model_type}' not supported for WOA13 obs")

    # Load WOA13 observations on the matching grid (cached)
    dsobs = load_woa13(obs_key)

    # Time-average model data (lazy evaluation)
    thetao_mean = ds_t.thetao.mean("time")
    so_mean = ds_s.so.mean("time")

    # Compute bias (model - obs), squeeze singleton dims
    tdiff = (thetao_mean - dsobs.thetao).squeeze()
    sdiff = (so_mean - dsobs.so).squeeze()

    # Get grid for depth and basin masks
    grid = momgrid.MOMgrid(model_type, warn=False).to_xarray()
    basins = cmip_basins.generate_basin_codes(grid)

    # Depth interfaces: try z_i from dataset, fall back to infer_bounds
    if "z_i" in ds_t.coords:
        z_i = ds_t.z_i
    else:
        z_i = xr.DataArray(
            momgrid.util.infer_bounds(ds_t.z_l.values), dims=("z_i",)
        )

    data[group] = {
        "tdiff": tdiff,
        "sdiff": sdiff,
        "depth": grid.deptho,
        "basins": basins,
        "z_l": ds_t.z_l,
        "z_i": z_i,
        "y": ds_t.geolat.mean("xh"),
        "areacello": ds_t.areacello,
    }

In [ ]:
# Define contour levels (same for all basins, matching original)
t_levels = np.array(
    [-2., -1.75, -1.5, -1.25]
    + list(np.arange(-0.9, 1.1, 0.2))
    + [1.25, 1.5, 1.75, 2.]
)
s_levels = t_levels.copy()

for basin_name, basin_cfg in BASINS.items():
    nexps = len(diag.groups)
    
    # Create two separate figures: one for temperature and one for salinity
    for varname, levels, cmap_name, unit_label in [
        ("thetao", t_levels, "RdBu_r", "Temperature Bias [degC]"),
        ("so", s_levels, "BrBG_r", "Salinity Bias [PSU]"),
    ]:
        # Use nbtools to get appropriate figure size and subplot layout
        figsize, subplot = nbtools.get_figsize_subplots(nexps)
        # Adjust figure height for y-z plots
        figsize = (figsize[0], figsize[1] * 1.2)
        fig, axes = plt.subplots(*subplot, figsize=figsize, dpi=150)
        axes = np.atleast_1d(axes).flatten()

        for n, group in enumerate(diag.groups):
            d = data[group]

            # Basin mask
            if basin_cfg["codes"] is not None:
                mask = xr.where(
                    sum(d["basins"] == c for c in basin_cfg["codes"]) > 0,
                    1, np.nan,
                )
            else:
                mask = 1.0

            # Zonal-mean bias
            if varname == "thetao":
                diff_zm = (d["tdiff"] * mask).weighted(d["areacello"]).mean("xh")
            else:
                diff_zm = (d["sdiff"] * mask).weighted(d["areacello"]).mean("xh")
            
            depth_max = (d["depth"] * mask).max("xh") if basin_cfg["codes"] is not None else d["depth"].max("xh")
            z = d["z_l"]
            y = d["y"]

            ax = axes[n]
            ax.set_facecolor("gray")

            cmap, norm, _ = climplot.discrete_cmap(
                -2, 2, 0.25, cmap_name=cmap_name, center_on_white=True,
            )
            cb = ax.pcolormesh(y, z, diff_zm, shading="auto", cmap=cmap, norm=norm)
            ax.fill_between(y, 6750, depth_max, color="gray")
            ax.set_yscale("splitscale", zval=[6500, 1500, 0])
            ax.hlines(1500, y.min(), y.max(), colors="k", linestyles="dashed", linewidths=0.5)
            ax.set_yticks([300, 600, 900, 1200, 1500, 2500, 3500, 4500, 5500, 6500])
            ax.set_xticks([-80, -60, -40, -20, 0, 20, 40, 60])

            if basin_cfg["xlim"] is not None:
                ax.set_xlim(*basin_cfg["xlim"])

            stats, stats_str = yz_stats(diff_zm, y, d["z_i"])
            ax.text(0.99, 1.03, stats_str, ha="right", transform=ax.transAxes, fontsize=7)
            ax.set_title(f"{group.name}", fontsize=8)
            ax.set_ylabel("Depth [m]", fontsize=7)
            ax.set_xlabel("Latitude", fontsize=7)

            # Register metrics
            basin_key = basin_name.lower().replace("+", "").replace(" ", "_")
            for k, v in stats.items():
                group.add_metric(f"{varname}_{basin_key}", (k, float(v)))

        # Hide any unused subplots
        for idx in range(nexps, len(axes)):
            axes[idx].axis('off')

        # Single colorbar at figure bottom
        cbar = nbtools.bottom_colorbar(fig, cb, orientation="horizontal", extend="both")
        cbar.set_label(unit_label, fontsize=7)

        var_title = "Temperature" if varname == "thetao" else "Salinity"
        fig.suptitle(f"{basin_name} - {var_title} Bias", y=1.01)
        all_figs.append(fig)

### Write Metrics to File

In [ ]:
diag.write_metrics("WOA13_metrics.json")

### Make a PowerPoint Presentation of Figures

In [ ]:
nbtools.save_pptx(all_figs, "WOA13_yz_bias.pptx")